In [1]:
import os, gc, random, shutil, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import torchaudio.transforms as T
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import ASTFeatureExtractor, ASTModel

warnings.filterwarnings('ignore')

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_all(42)

n_gpu = torch.cuda.device_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if n_gpu > 0:
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_memory/1e9:.1f}GB)")
else:
    print("No GPU found, using CPU")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

print("Hugging Face Authentication Successful!")

2026-03-21 17:31:22.544646: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774114282.908275      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774114283.009717      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774114283.989022      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774114283.989070      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774114283.989073      24 computation_placer.cc:177] computation placer alr

GPU 0: Tesla T4 (15.6GB)
GPU 1: Tesla T4 (15.6GB)
Hugging Face Authentication Successful!


In [2]:
class CFG:
    root = Path('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup')
    stems = root / 'genres_stems'
    noise = root / 'ESC-50-master' / 'audio'
    mashups = root / 'mashups'
    test_csv = root / 'test.csv'
    out = Path('/kaggle/working')
    out.mkdir(exist_ok=True)
    
    sr = 16000
    duration = 10
    n_samples = sr * duration
    full_dur = 30
    full_samples = sr * full_dur
    
    ast_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
    
    n_mels = 128
    target_len = 1024
    
    bs = 48 if n_gpu >= 2 else 24
    epochs = 25
    workers = 4
    lr = 3e-5
    wd = 0.02
    warmup = 0.05
    grad_clip = 1.0
    
    patchout_t = 0.4
    patchout_f = 0.2
    
    snr_lo, snr_hi = 10, 28
    vol_lo, vol_hi = 0.8, 1.2
    mixup_p = 0.3
    mixup_a = 0.3
    
    fm_param = 24
    tm_param = 48
    n_fm = 2
    n_tm = 2
    spec_aug_p = 0.6
    
    tta_crops = 7
    
    genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
    n_cls = len(genres)
    g2i = {g: i for i, g in enumerate(genres)}
    i2g = {i: g for i, g in enumerate(genres)}
    
    train_samples = 10000
    val_samples = 2000
    
    use_amp = True
    patience = 8
    multi_gpu = n_gpu >= 2
    
    label_smooth = 0.05

cfg = CFG()

print(f"Config: bs={cfg.bs}, epochs={cfg.epochs}, lr={cfg.lr}")
print(f"Patchout: time={cfg.patchout_t}, freq={cfg.patchout_f}")

Config: bs=48, epochs=25, lr=3e-05
Patchout: time=0.4, freq=0.2


In [3]:
class StemCache:
    def __init__(self, stems_path, noise_path, sr=16000):
        self.sr = sr
        self.data = {}
        self.noises = []
        
        print("Caching audio...")
        self._cache_stems(stems_path)
        self._cache_noise(noise_path)
        
        n_stems = sum(len(songs) for songs in self.data.values())
        print(f"Cached: {n_stems} songs, {len(self.noises)} noise clips")
    
    def _load(self, p, dur=None):
        try:
            y, _ = librosa.load(p, sr=self.sr, duration=dur, mono=True)
            return y.astype(np.float32)
        except:
            return np.zeros(self.sr * 5, dtype=np.float32)
    
    def _cache_stems(self, path):
        stems_list = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        
        for gdir in tqdm(sorted(path.iterdir()), desc="Stems"):
            if not gdir.is_dir() or gdir.name not in cfg.genres:
                continue
            
            self.data[gdir.name] = {}
            
            for sdir in sorted(gdir.iterdir()):
                if not sdir.is_dir():
                    continue
                
                song_stems = {}
                for st in stems_list:
                    sp = sdir / st
                    if sp.exists():
                        song_stems[st] = self._load(sp, dur=30)
                
                if song_stems:
                    self.data[gdir.name][sdir.name] = song_stems
    
    def _cache_noise(self, path):
        for nf in tqdm(list(path.glob('*.wav')), desc="Noise"):
            audio = self._load(nf, dur=5)
            if len(audio) > 1000:
                self.noises.append(audio)
    
    def get_stems(self, genre):
        if genre not in self.data or len(self.data[genre]) < 4:
            return {}
        
        songs = list(self.data[genre].keys())
        picked = random.sample(songs, 4)
        stem_names = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
        
        out = {}
        for i, sn in enumerate(stem_names):
            sid = picked[i % len(picked)]
            if sn in self.data[genre][sid]:
                out[sn] = self.data[genre][sid][sn].copy()
        return out
    
    def get_noise(self):
        if not self.noises:
            return np.zeros(self.sr * 5, dtype=np.float32)
        return random.choice(self.noises).copy()

cache = StemCache(cfg.stems, cfg.noise, cfg.sr)
gc.collect()

Caching audio...


Stems:   0%|          | 0/10 [00:00<?, ?it/s]

Noise:   0%|          | 0/2000 [00:00<?, ?it/s]

Cached: 1000 songs, 2000 noise clips


31401

In [4]:
class AudioUtils:
    @staticmethod
    def normalize(x):
        if len(x) == 0:
            return x
        mx = np.abs(x).max()
        return x / mx * 0.95 if mx > 0 else x
    
    @staticmethod
    def rms_norm(x, db=-20.0):
        if len(x) == 0:
            return x
        rms = np.sqrt(np.mean(x**2))
        if rms < 1e-8:
            return x
        target = 10 ** (db / 20)
        return np.clip(x * (target / rms), -1.0, 1.0)
    
    @staticmethod
    def add_noise(clean, noise, snr):
        if len(noise) < len(clean):
            reps = int(np.ceil(len(clean) / len(noise)))
            noise = np.tile(noise, reps)[:len(clean)]
        else:
            st = random.randint(0, max(0, len(noise) - len(clean)))
            noise = noise[st:st + len(clean)]
        
        cp = np.mean(clean**2) + 1e-10
        np_ = np.mean(noise**2) + 1e-10
        scale = np.sqrt(cp / (10 ** (snr / 10)) / np_)
        
        return AudioUtils.normalize(clean + noise * scale)
    
    @staticmethod
    def pad_trim(x, length):
        if len(x) < length:
            return np.pad(x, (0, length - len(x)))
        return x[:length]
    
    @staticmethod
    def rand_crop(x, length):
        if len(x) <= length:
            return AudioUtils.pad_trim(x, length)
        st = random.randint(0, len(x) - length)
        return x[st:st + length]

au = AudioUtils()

In [5]:
class ASTProcessor:
    def __init__(self, sr=16000):
        self.sr = sr
        self.fe = ASTFeatureExtractor.from_pretrained(cfg.ast_name, sampling_rate=sr)
    
    def __call__(self, audio):
        inp = self.fe(audio, sampling_rate=self.sr, return_tensors="pt")
        return inp.input_values.squeeze(0)

ast_proc = ASTProcessor(cfg.sr)

class SpecAug:
    def __init__(self, fm=24, tm=48, nf=2, nt=2):
        self.fm = fm
        self.tm = tm
        self.nf = nf
        self.nt = nt
    
    def __call__(self, spec):
        if spec.dim() == 2:
            spec = spec.unsqueeze(0)
            sq = True
        else:
            sq = False
        
        b, t, f = spec.shape
        spec = spec.clone()
        
        for _ in range(self.nf):
            fw = random.randint(1, min(self.fm, f-1))
            f0 = random.randint(0, f - fw)
            spec[:, :, f0:f0+fw] = 0
        
        for _ in range(self.nt):
            tw = random.randint(1, min(self.tm, t-1))
            t0 = random.randint(0, t - tw)
            spec[:, t0:t0+tw, :] = 0
        
        return spec.squeeze(0) if sq else spec

spec_aug = SpecAug(cfg.fm_param, cfg.tm_param, cfg.n_fm, cfg.n_tm)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [6]:
class MashupDS(Dataset):
    def __init__(self, cache, n_samples, is_val=False, aug=True):
        self.cache = cache
        self.n = n_samples
        self.is_val = is_val
        self.aug = aug and not is_val
        self.crop_len = cfg.n_samples
        self.full_len = cfg.full_samples
    
    def __len__(self):
        return self.n
    
    def _make_mashup(self, genre):
        stems = self.cache.get_stems(genre)
        if not stems:
            return np.zeros(self.full_len, dtype=np.float32)
        
        mix = np.zeros(self.full_len, dtype=np.float32)
        
        for name, audio in stems.items():
            audio = au.pad_trim(audio, self.full_len)
            vol = random.uniform(cfg.vol_lo, cfg.vol_hi) if self.aug else 1.0
            mix += audio * vol
        
        mix = au.rms_norm(mix, -20.0)
        
        if self.aug:
            for _ in range(random.randint(1, 2)):
                noise = self.cache.get_noise()
                snr = random.uniform(cfg.snr_lo, cfg.snr_hi)
                
                nf = np.zeros(self.full_len, dtype=np.float32)
                noise = au.pad_trim(noise, len(noise))
                
                if len(noise) < self.full_len:
                    st = random.randint(0, self.full_len - len(noise))
                    nf[st:st+len(noise)] = noise
                else:
                    nf = noise[:self.full_len]
                
                mix = au.add_noise(mix, nf, snr)
        
        return mix
    
    def __getitem__(self, idx):
        genre = random.choice(cfg.genres)
        label = cfg.g2i[genre]
        
        audio = self._make_mashup(genre)
        crop = au.rand_crop(audio, self.crop_len)
        
        feat = ast_proc(crop)
        
        if self.aug and random.random() < cfg.spec_aug_p:
            feat = spec_aug(feat)
        
        lbl = torch.zeros(cfg.n_cls, dtype=torch.float32)
        
        if self.aug and random.random() < cfg.mixup_p:
            g2 = random.choice(cfg.genres)
            l2 = cfg.g2i[g2]
            
            a2 = self._make_mashup(g2)
            c2 = au.rand_crop(a2, self.crop_len)
            f2 = ast_proc(c2)
            
            if random.random() < cfg.spec_aug_p:
                f2 = spec_aug(f2)
            
            lam = np.random.beta(cfg.mixup_a, cfg.mixup_a)
            feat = lam * feat + (1 - lam) * f2
            
            lbl[label] = lam
            lbl[l2] += (1 - lam)
        else:
            if cfg.label_smooth > 0 and self.aug:
                lbl.fill_(cfg.label_smooth / cfg.n_cls)
                lbl[label] = 1.0 - cfg.label_smooth + cfg.label_smooth / cfg.n_cls
            else:
                lbl[label] = 1.0
        
        return feat, lbl

print("Testing dataset...")
test_ds = MashupDS(cache, 3, is_val=False)
for i in range(2):
    f, l = test_ds[i]
    print(f"  Sample {i}: {f.shape}, label_sum={l.sum():.3f}")
del test_ds

Testing dataset...
  Sample 0: torch.Size([1024, 128]), label_sum=1.000
  Sample 1: torch.Size([1024, 128]), label_sum=1.000


In [7]:
class Patchout(nn.Module):
    def __init__(self, p_time=0.4, p_freq=0.2):
        super().__init__()
        self.p_t = p_time
        self.p_f = p_freq
    
    def forward(self, x):
        if not self.training:
            return x
        
        b, t, f = x.shape
        mask = torch.ones_like(x)
        
        if self.p_t > 0 and random.random() < 0.5:
            n_drop = int(t * self.p_t * random.random())
            if n_drop > 0:
                indices = random.sample(range(t), n_drop)
                mask[:, indices, :] = 0
        
        if self.p_f > 0 and random.random() < 0.5:
            n_drop = int(f * self.p_f * random.random())
            if n_drop > 0:
                indices = random.sample(range(f), n_drop)
                mask[:, :, indices] = 0
        
        return x * mask


class ASTClassifier(nn.Module):
    def __init__(self, n_cls=10, pretrained=True, patchout_t=0.4, patchout_f=0.2):
        super().__init__()
        
        self.patchout = Patchout(patchout_t, patchout_f)
        self.ast = ASTModel.from_pretrained(cfg.ast_name) if pretrained else ASTModel.from_pretrained(cfg.ast_name)
        self.hidden = self.ast.config.hidden_size
        
        self.head = nn.Sequential(
            nn.LayerNorm(self.hidden),
            nn.Dropout(0.15),
            nn.Linear(self.hidden, 384),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(384, n_cls)
        )
        
        self._init_head()
    
    def _init_head(self):
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.patchout(x)
        out = self.ast(x)
        pooled = out.pooler_output
        return self.head(pooled)


print("Testing model...")
model = ASTClassifier(cfg.n_cls, patchout_t=cfg.patchout_t, patchout_f=cfg.patchout_f).to(device)

dummy = torch.randn(2, 1024, 128).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Model output: {out.shape}")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

del model, dummy, out
torch.cuda.empty_cache()

Testing model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Model output: torch.Size([2, 10])
Trainable params: 86,487,946


In [8]:
class SoftCE(nn.Module):
    def forward(self, logits, targets):
        lp = F.log_softmax(logits, dim=-1)
        return -(targets * lp).sum(dim=-1).mean()

loss_fn = SoftCE()


class Trainer:
    def __init__(self, model, train_dl, val_dl, dev):
        self.dev = dev
        
        if cfg.multi_gpu and n_gpu > 1:
            self.model = nn.DataParallel(model)
            print(f"Using {n_gpu} GPUs")
        else:
            self.model = model
        
        self.model = self.model.to(dev)
        self.train_dl = train_dl
        self.val_dl = val_dl
        
        self.opt = torch.optim.AdamW(
            self.model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.wd,
            betas=(0.9, 0.98)
        )
        
        total_steps = len(train_dl) * cfg.epochs
        warmup_steps = int(total_steps * cfg.warmup)
        
        self.sched = torch.optim.lr_scheduler.OneCycleLR(
            self.opt,
            max_lr=cfg.lr,
            total_steps=total_steps,
            pct_start=cfg.warmup,
            anneal_strategy='cos',
            div_factor=25,
            final_div_factor=1000
        )
        
        self.scaler = GradScaler() if cfg.use_amp else None
        
        self.best_f1 = 0.0
        self.best_ep = 0
        self.wait = 0
        self.history = {'tr_loss': [], 'val_loss': [], 'val_f1': []}
    
    def train_one(self, ep):
        self.model.train()
        tot_loss = 0
        n_batch = 0
        
        pbar = tqdm(self.train_dl, desc=f"Ep {ep+1}/{cfg.epochs}")
        
        for feats, tgts in pbar:
            feats = feats.to(self.dev)
            tgts = tgts.to(self.dev)
            
            self.opt.zero_grad()
            
            if cfg.use_amp:
                with autocast():
                    logits = self.model(feats)
                    loss = loss_fn(logits, tgts)
                
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.opt)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.grad_clip)
                self.scaler.step(self.opt)
                self.scaler.update()
            else:
                logits = self.model(feats)
                loss = loss_fn(logits, tgts)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.grad_clip)
                self.opt.step()
            
            self.sched.step()
            tot_loss += loss.item()
            n_batch += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg': f'{tot_loss/n_batch:.4f}'})
        
        return tot_loss / n_batch
    
    def validate(self):
        self.model.eval()
        tot_loss = 0
        preds, truths = [], []
        
        with torch.no_grad():
            for feats, tgts in tqdm(self.val_dl, desc="Val"):
                feats = feats.to(self.dev)
                tgts = tgts.to(self.dev)
                
                if cfg.use_amp:
                    with autocast():
                        logits = self.model(feats)
                        loss = loss_fn(logits, tgts)
                else:
                    logits = self.model(feats)
                    loss = loss_fn(logits, tgts)
                
                tot_loss += loss.item()
                preds.extend(logits.argmax(1).cpu().numpy())
                truths.extend(tgts.argmax(1).cpu().numpy())
        
        avg_loss = tot_loss / len(self.val_dl)
        macro_f1 = f1_score(truths, preds, average='macro')
        per_cls = f1_score(truths, preds, average=None)
        
        return avg_loss, macro_f1, per_cls
    
    def save(self, path):
        state = self.model.module.state_dict() if hasattr(self.model, 'module') else self.model.state_dict()
        torch.save({'state': state, 'f1': self.best_f1, 'epoch': self.best_ep}, path)
    
    def run(self):
        print(f"\nTraining: {len(self.train_dl.dataset)} samples, {cfg.epochs} epochs")
        
        for ep in range(cfg.epochs):
            tr_loss = self.train_one(ep)
            val_loss, val_f1, pc_f1 = self.validate()
            
            self.history['tr_loss'].append(tr_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_f1'].append(val_f1)
            
            print(f"Ep {ep+1}: tr_loss={tr_loss:.4f}, val_loss={val_loss:.4f}, F1={val_f1:.4f}")
            print("  " + " ".join(f"{cfg.genres[i][:3]}:{pc_f1[i]:.3f}" for i in range(cfg.n_cls)))
            
            if val_f1 > self.best_f1:
                self.best_f1 = val_f1
                self.best_ep = ep
                self.wait = 0
                self.save(cfg.out / 'best.pth')
                print(f"  -> New best: {val_f1:.4f}")
            else:
                self.wait += 1
                if self.wait >= cfg.patience:
                    print(f"Early stop at ep {ep+1}")
                    break
        
        print(f"\nDone! Best F1: {self.best_f1:.4f} at ep {self.best_ep+1}")
        return self.best_f1

In [9]:
seed_all(42)

train_ds = MashupDS(cache, cfg.train_samples, is_val=False, aug=True)
val_ds = MashupDS(cache, cfg.val_samples, is_val=True, aug=False)

train_dl = DataLoader(train_ds, batch_size=cfg.bs, shuffle=True, num_workers=cfg.workers,
                      pin_memory=True, persistent_workers=True, prefetch_factor=2)
val_dl = DataLoader(val_ds, batch_size=cfg.bs, shuffle=False, num_workers=cfg.workers, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"Batches: train={len(train_dl)}, val={len(val_dl)}")

model = ASTClassifier(cfg.n_cls, pretrained=True, patchout_t=cfg.patchout_t, patchout_f=cfg.patchout_f)

trainer = Trainer(model, train_dl, val_dl, device)
best_f1 = trainer.run()

del train_ds, val_ds, train_dl, val_dl
torch.cuda.empty_cache()
gc.collect()

Train: 10000, Val: 2000
Batches: train=209, val=42
Using 2 GPUs

Training: 10000 samples, 25 epochs


Ep 1/25:   0%|          | 0/209 [00:00<?, ?it/s]

Val:   0%|          | 0/42 [00:00<?, ?it/s]

Ep 1: tr_loss=1.6338, val_loss=0.4831, F1=0.9077
  blu:0.904 cla:0.992 cou:0.851 dis:0.900 hip:0.938 jaz:0.975 met:0.958 pop:0.885 reg:0.904 roc:0.768
  -> New best: 0.9077


Ep 2/25:   0%|          | 0/209 [00:00<?, ?it/s]

...

Ep 24: tr_loss=0.4008, val_loss=0.0495, F1=0.9975
  blu:1.000 cla:1.000 cou:0.995 dis:0.997 hip:0.995 jaz:1.000 met:0.998 pop:0.998 reg:1.000 roc:0.993


Ep 25/25:   0%|          | 0/209 [00:00<?, ?it/s]

Val:   0%|          | 0/42 [00:00<?, ?it/s]

Ep 25: tr_loss=0.3852, val_loss=0.0476, F1=0.9985
  blu:0.997 cla:1.000 cou:0.993 dis:1.000 hip:1.000 jaz:1.000 met:1.000 pop:1.000 reg:1.000 roc:0.995
  -> New best: 0.9985

Done! Best F1: 0.9985 at ep 25


333

In [10]:
class TestDS(Dataset):
    def __init__(self, mashup_dir, test_df, n_crops=7):
        self.mdir = mashup_dir
        self.df = test_df
        self.n_crops = n_crops
        self.crop_len = cfg.n_samples
        self.full_len = cfg.full_samples
    
    def __len__(self):
        return len(self.df)
    
    def _load(self, aid):
        fname = f"song{str(aid).zfill(4)}.wav"
        path = self.mdir / fname
        
        try:
            y, _ = librosa.load(path, sr=cfg.sr, duration=30, mono=True)
            y = au.pad_trim(y, self.full_len)
            y = au.normalize(y)
            return y.astype(np.float32)
        except Exception as e:
            return np.zeros(self.full_len, dtype=np.float32)
    
    def _crops(self, audio):
        crops = []
        total = len(audio)
        clen = self.crop_len
        
        if total <= clen:
            audio = au.pad_trim(audio, clen)
            f = ast_proc(audio)
            return [f] * self.n_crops
        
        max_st = total - clen
        starts = np.linspace(0, max_st, self.n_crops, dtype=int)
        
        for st in starts:
            crop = audio[st:st+clen]
            crops.append(ast_proc(crop))
        
        return crops
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        aid = row['id']
        
        audio = self._load(aid)
        crops = self._crops(audio)
        
        return str(aid), torch.stack(crops)


def inference(model, test_dl):
    model.eval()
    preds = {}
    probs = {}
    
    with torch.no_grad():
        for aids, crops in tqdm(test_dl, desc="Inference"):
            bs, nc = crops.shape[:2]
            flat = crops.view(-1, *crops.shape[2:]).to(device)
            
            if cfg.use_amp:
                with autocast():
                    logits = model(flat)
            else:
                logits = model(flat)
            
            p = F.softmax(logits, dim=1)
            p = p.view(bs, nc, -1).mean(dim=1)
            
            pred_idx = p.argmax(dim=1)
            
            for i, aid in enumerate(aids):
                preds[aid] = cfg.i2g[pred_idx[i].item()]
                probs[aid] = p[i].cpu().numpy()
    
    return preds, probs

In [11]:
print("Loading test data...")
test_df = pd.read_csv(cfg.test_csv)
print(f"Test samples: {len(test_df)}")

test_ds = TestDS(cfg.mashups, test_df, n_crops=cfg.tta_crops)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Loading best model...")
model = ASTClassifier(cfg.n_cls, pretrained=False, patchout_t=0, patchout_f=0)
ckpt = torch.load(cfg.out / 'best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['state'])
model = model.to(device)
print(f"Loaded model with F1={ckpt['f1']:.4f}")

print("Running inference...")
predictions, all_probs = inference(model, test_dl)

sub = pd.DataFrame([{'id': k, 'genre': v} for k, v in predictions.items()])
sub['id'] = sub['id'].astype(int)
sub = sub.sort_values('id')
sub['id'] = sub['id'].astype(str).str.zfill(4)

sub_path = cfg.out / 'submission.csv'
sub.to_csv(sub_path, index=False)

print(f"\nSaved: {sub_path}")
print(f"Predictions: {len(sub)}")
print("\nDistribution:")
print(sub['genre'].value_counts().sort_index())

prob_df = pd.DataFrame([{'id': k, **{cfg.genres[i]: v[i] for i in range(len(v))}} for k, v in all_probs.items()])
prob_df['id'] = prob_df['id'].astype(int)
prob_df = prob_df.sort_values('id')
prob_df['id'] = prob_df['id'].astype(str).str.zfill(4)
prob_df.to_csv(cfg.out / 'probs.csv', index=False)

Loading test data...
Test samples: 3020
Loading best model...
Loaded model with F1=0.9985
Running inference...


Inference:   0%|          | 0/378 [00:00<?, ?it/s]


Saved: /kaggle/working/submission.csv
Predictions: 3020

Distribution:
genre
blues        307
classical    255
country      275
disco        308
hiphop       308
jazz         302
metal        319
pop          310
reggae       309
rock         327
Name: count, dtype: int64
